# Phase 0c: Modeling experiments
Covers what `01_model_prototype` skipped: richer feature engineering, a model comparison, feature selection, hyperparameter tuning, and class imbalance.

**Three tracks, one harness**
- `ret`: regress the next trading day's log return (baseline: predict zero)
- `vol`: regress the next day's absolute return, i.e. volatility (baseline: rolling 20-day std x 0.7979)
- `direction`: classify up vs down (about 50/50, so **no imbalance handling is needed**)
- `event`: classify a "large move" day (top 10% of |return|), a separate spike-alert question that IS imbalanced

**Leakage rules:** features use information through day d-1 only. Anything fitted (clip bounds, event thresholds, feature selection, scalers) is fitted on the training part of each CV fold only. Test is looked at once, at the end.

**Honesty note:** the `vol` track was suggested after a quick single-run look at the test window. Weigh the CV numbers more than the final test numbers for that track.

## 1. Setup and prep (same cleaning rules as `00_eda`)

In [ ]:
import time, warnings
import polars as pl
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA = "../data/raw/daily_forex_rates.csv"
PROTO = ["USD", "GBP", "JPY", "CHF", "AUD", "CAD", "LKR"]
SPLIT_FRAC = 0.8
THR, REVERSAL = 0.05, 0.3

raw = (
    pl.read_csv(DATA, try_parse_dates=True)
      .filter(pl.col("date").dt.weekday() <= 5)
      .sort(["currency", "date"])
)
all_dates = raw["date"].unique().sort()
CUT = all_dates[int(len(all_dates) * SPLIT_FRAC)]
print("train: dates <", CUT, "| test: dates >=", CUT)

d = (
    raw.filter(pl.col("currency").is_in(PROTO))
       .with_columns(rate_raw=pl.col("exchange_rate"))
       .with_columns(ret_raw=pl.col("rate_raw").log().diff().over("currency"))
       .with_columns(nxt=pl.col("ret_raw").shift(-1).over("currency"))
)
glitch = (pl.col("ret_raw").abs() > THR) & (
    (pl.col("ret_raw") + pl.col("nxt")).abs() < REVERSAL * pl.col("ret_raw").abs()
)
d = (
    d.with_columns(is_glitch=glitch.fill_null(False))
     .with_columns(
         rate=pl.when(pl.col("is_glitch")).then(None).otherwise(pl.col("rate_raw"))
                .fill_null(strategy="forward").over("currency")
     )
     .with_columns(log_ret=pl.col("rate").log().diff().over("currency"))
     .drop(["nxt", "exchange_rate"])
)
print(d.shape, "| glitch rows among prototypes:", d["is_glitch"].sum())

c:\Users\Acer\Documents\GitHub\forex-engine\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train: dates < 2022-05-06 | test: dates >= 2022-05-06
(37558, 9) | glitch rows among prototypes: 2


## 2. Feature engineering v2
Groups (all per currency, all using the series shifted by one day):
- **Return lags** 1-30 and **absolute-return lags** 1-5 (volatility clustering)
- **Rolling** mean, std, mean-|return| and momentum (sum of returns) over 5/10/20/30 days
- **EWMA volatility** (spans 10 and 30)
- **Price deviation**: how far the log rate sits from its 20/60-day mean, in std units (mean reversion)
- **Market features**: cross-currency average of yesterday's return, |return| and volatility on the same date
- **Calendar**: weekday, month, day of month, month-end flag

This is a candidate pool; section 5 prunes it.

In [ ]:
LAGS = list(range(1, 31))
WINDOWS = [5, 10, 20, 30]

def build_features(frame: pl.DataFrame) -> pl.DataFrame:
    r = pl.col("log_ret")
    past = r.shift(1)                          # information through day d-1
    lrate = pl.col("rate").log().shift(1)

    exprs = [r.shift(k).over("currency").alias(f"lag_{k}") for k in LAGS]
    exprs += [r.shift(k).abs().over("currency").alias(f"abs_lag_{k}") for k in range(1, 6)]
    for w in WINDOWS:
        exprs += [
            past.rolling_mean(window_size=w).over("currency").alias(f"roll_mean_{w}"),
            past.rolling_std(window_size=w).over("currency").alias(f"roll_std_{w}"),
            past.abs().rolling_mean(window_size=w).over("currency").alias(f"roll_abs_{w}"),
            past.rolling_sum(window_size=w).over("currency").alias(f"mom_{w}"),
        ]
    for span in (10, 30):
        exprs.append((past ** 2).ewm_mean(span=span, ignore_nulls=True).sqrt().over("currency").alias(f"ewm_vol_{span}"))
    for w in (20, 60):
        exprs.append(
            ((lrate - lrate.rolling_mean(window_size=w)) / lrate.rolling_std(window_size=w))
            .over("currency").alias(f"dev_{w}")
        )

    out = frame.with_columns(exprs).with_columns(
        vol_ratio=pl.col("roll_std_5") / pl.col("roll_std_30"),
        dow=pl.col("date").dt.weekday(),
        month=pl.col("date").dt.month(),
        dom=pl.col("date").dt.day(),
        month_end=(pl.col("date").dt.day() >= 28).cast(pl.Int8),
    ).with_columns(
        mkt_lag1=pl.col("lag_1").mean().over("date"),
        mkt_abs_lag1=pl.col("abs_lag_1").mean().over("date"),
        mkt_vol_5=pl.col("roll_std_5").mean().over("date"),
    )
    return out

NUM = (
    [f"lag_{k}" for k in LAGS] + [f"abs_lag_{k}" for k in range(1, 6)]
    + [f"{p}_{w}" for p in ("roll_mean", "roll_std", "roll_abs", "mom") for w in WINDOWS]
    + ["ewm_vol_10", "ewm_vol_30", "dev_20", "dev_60", "vol_ratio",
       "dow", "month", "dom", "month_end", "mkt_lag1", "mkt_abs_lag1", "mkt_vol_5"]
)
ALL = ["currency"] + NUM

feats = (
    build_features(d)
    .drop_nulls(NUM + ["log_ret", "ret_raw"])
    .filter(pl.all_horizontal(pl.col(NUM).is_finite()))
    .with_columns(
        y_ret=pl.col("log_ret"),                      # training targets (repaired)
        y_vol=pl.col("log_ret").abs(),
        e_ret=pl.col("ret_raw"),                      # evaluation targets (raw)
        e_vol=pl.col("ret_raw").abs(),
        base_vol=pl.col("roll_std_20") * 0.7979,      # naive volatility forecast
    )
    .sort(["date", "currency"])
)
train_df = feats.filter(pl.col("date") < CUT)
test_df = feats.filter(pl.col("date") >= CUT)
print("candidate features:", len(ALL), "| train rows:", train_df.height, "| test rows:", test_df.height)

chk = feats.filter(pl.col("currency") == "USD").with_columns(prev=pl.col("log_ret").shift(1)).drop_nulls("prev")
print("lag_1 == previous day's return:", bool(((chk["lag_1"] - chk["prev"]).abs() < 1e-12).all()))

candidate features: 64 | train rows: 29083 | test rows: 8055
lag_1 == previous day's return: True


## 3. Shared harness: folds, clipping, models, metrics
Expanding-window folds over unique trading dates, computed once and reused everywhere. Clip bounds are refitted per fold from that fold's training part. Metrics use the raw target; `ratio` is MAE divided by the baseline's MAE (below 1.0 beats the baseline).

In [ ]:
Q_LO, Q_HI = 0.005, 0.995
TASKS = ["ret", "vol"]

def make_folds(df: pl.DataFrame, n_splits: int = 5):
    dates = df["date"].unique().sort().to_list()
    out = []
    for tr_i, va_i in TimeSeriesSplit(n_splits=n_splits).split(dates):
        tr = df.filter(pl.col("date") <= dates[tr_i[-1]])
        va = df.filter((pl.col("date") >= dates[va_i[0]]) & (pl.col("date") <= dates[va_i[-1]]))
        out.append((tr, va))
    return out

FOLDS = make_folds(train_df)
print("folds:", [(a.height, b.height) for a, b in FOLDS])

def fit_clip_bounds(train: pl.DataFrame, col: str) -> pl.DataFrame:
    return train.group_by("currency").agg(
        lo=pl.col(col).quantile(Q_LO), hi=pl.col(col).quantile(Q_HI)
    )

def clipped(frame: pl.DataFrame, bounds: pl.DataFrame, col: str) -> np.ndarray:
    j = frame.join(bounds, on="currency", how="left")
    return j.select(pl.col(col).clip(pl.col("lo"), pl.col("hi")))[col].to_numpy()

def make_X_lgb(frame, cols):
    X = frame.select(cols).to_pandas()
    if "currency" in cols:
        X["currency"] = pd.Categorical(X["currency"], categories=sorted(PROTO))
    return X

def make_X_sk(frame, cols):
    X = frame.select([c for c in cols if c != "currency"]).to_pandas()
    for c in sorted(PROTO):
        X[f"ccy_{c}"] = (frame["currency"] == c).to_numpy().astype(float)
    return X

LGB_BASE = dict(objective="l1", n_estimators=300, learning_rate=0.03, num_leaves=15,
                min_child_samples=100, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                reg_lambda=1.0, random_state=42, verbose=-1, n_jobs=-1)

MODELS = {
    "ridge": dict(
        make=lambda p=None: make_pipeline(StandardScaler(), Ridge(alpha=100.0)),
        X=make_X_sk, y="clipped"),
    "random_forest": dict(
        make=lambda p=None: RandomForestRegressor(n_estimators=60, min_samples_leaf=50, max_features=0.3,
                                                  max_samples=0.5, n_jobs=-1, random_state=42),
        X=make_X_sk, y="clipped"),
    "extra_trees": dict(
        make=lambda p=None: ExtraTreesRegressor(n_estimators=60, min_samples_leaf=50, max_features=0.3,
                                                max_samples=0.5, bootstrap=True, n_jobs=-1, random_state=42),
        X=make_X_sk, y="clipped"),
    "lightgbm": dict(
        make=lambda p=None: lgb.LGBMRegressor(**{**LGB_BASE, **(p or {})}),
        X=make_X_lgb, y="raw"),
    "xgboost": dict(
        make=lambda p=None: xgb.XGBRegressor(objective="reg:absoluteerror", n_estimators=300, learning_rate=0.03,
                                            max_depth=4, subsample=0.8, colsample_bytree=0.8,
                                            min_child_weight=50, tree_method="hist", n_jobs=-1,
                                            random_state=42, verbosity=0),
        X=make_X_sk, y="raw"),
}

def fit_predict(name, task, tr, va, cols=ALL, params=None):
    spec = MODELS[name]
    ycol = f"y_{task}"
    y = tr[ycol].to_numpy()
    if spec["y"] == "clipped":
        y = clipped(tr, fit_clip_bounds(tr, ycol), ycol)      # fitted on THIS training part only
    model = spec["make"](params)
    model.fit(spec["X"](tr, cols), y)
    return model.predict(spec["X"](va, cols)), model

def score(task, frame, pred):
    e = frame[f"e_{task}"].to_numpy()
    base = np.zeros(len(e)) if task == "ret" else frame["base_vol"].to_numpy()
    out = dict(mae=float(np.mean(np.abs(e - pred))), mae_base=float(np.mean(np.abs(e - base))))
    out["ratio"] = out["mae"] / out["mae_base"]
    if task == "ret":
        nz = e != 0
        out["dir_acc"] = float(np.mean(np.sign(pred[nz]) == np.sign(e[nz])))
    return out

folds: [(4524, 4554), (9078, 4554), (13632, 4925), (18557, 5282), (23839, 5244)]


## 4. Model comparison (default settings, all candidate features)
Ridge, Random Forest, Extra Trees, LightGBM, XGBoost, on both regression tracks. Linear and bagged-tree models train on the clipped target (squared-error losses are sensitive to extremes); LightGBM and XGBoost use an absolute-error loss on the repaired target.

In [ ]:
rows = []
for task in TASKS:
    for name in MODELS:
        t0 = time.time()
        for i, (tr, va) in enumerate(FOLDS):
            pred, _ = fit_predict(name, task, tr, va)
            rows.append(dict(task=task, model=name, fold=i, **score(task, va, pred)))
        print(f"{task:>4} {name:<14} done in {time.time() - t0:5.1f}s")

cmp = pl.DataFrame(rows)
summary = (
    cmp.group_by(["task", "model"])
       .agg(ratio=pl.col("ratio").mean(), ratio_std=pl.col("ratio").std(),
            mae=pl.col("mae").mean(), dir_acc=pl.col("dir_acc").mean())
       .sort(["task", "ratio"])
)
print(summary)

 ret ridge          done in   0.7s
 ret random_forest  done in   7.6s
 ret extra_trees    done in   1.0s
 ret lightgbm       done in   3.3s
 ret xgboost        done in   6.9s
 vol ridge          done in   0.4s
 vol random_forest  done in   4.1s
 vol extra_trees    done in   1.1s
 vol lightgbm       done in   3.8s
 vol xgboost        done in   6.2s
shape: (10, 6)
┌──────┬───────────────┬──────────┬───────────┬──────────┬──────────┐
│ task ┆ model         ┆ ratio    ┆ ratio_std ┆ mae      ┆ dir_acc  │
│ ---  ┆ ---           ┆ ---      ┆ ---       ┆ ---      ┆ ---      │
│ str  ┆ str           ┆ f64      ┆ f64       ┆ f64      ┆ f64      │
╞══════╪═══════════════╪══════════╪═══════════╪══════════╪══════════╡
│ ret  ┆ extra_trees   ┆ 1.000348 ┆ 0.000978  ┆ 0.004268 ┆ 0.502964 │
│ ret  ┆ random_forest ┆ 1.002921 ┆ 0.00198   ┆ 0.004278 ┆ 0.500265 │
│ ret  ┆ xgboost       ┆ 1.003414 ┆ 0.00376   ┆ 0.004283 ┆ 0.501409 │
│ ret  ┆ lightgbm      ┆ 1.003427 ┆ 0.004053  ┆ 0.004282 ┆ 0.51107  │
│ ret

## 5. Feature selection (inside each fold)
Four strategies, all decided from the fold's **training** part only, then scored with LightGBM on that fold's validation part:
- `all`: every candidate feature
- `corr_filter`: drop one of each pair with |correlation| > 0.95
- `top_20` / `top_10`: keep the features with the highest LightGBM gain

Selecting on the full training window and then cross-validating would leak, which is why this runs per fold.

In [ ]:
def gain_ranking(tr: pl.DataFrame, task: str) -> pd.Series:
    m = lgb.LGBMRegressor(**LGB_BASE).fit(make_X_lgb(tr, ALL), tr[f"y_{task}"].to_numpy())
    return pd.Series(m.booster_.feature_importance("gain"), index=ALL).sort_values(ascending=False)

def corr_filter(tr: pl.DataFrame, thr: float = 0.95) -> list:
    corr = tr.select(NUM).to_pandas().corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop = {c for c in upper.columns if (upper[c] > thr).any()}
    return ["currency"] + [c for c in NUM if c not in drop]

RANK = {(t, i): gain_ranking(tr, t) for t in TASKS for i, (tr, va) in enumerate(FOLDS)}
STRATS = {
    "all":         lambda tr, t, i: ALL,
    "corr_filter": lambda tr, t, i: corr_filter(tr),
    "top_20":      lambda tr, t, i: list(RANK[(t, i)].index[:20]),
    "top_10":      lambda tr, t, i: list(RANK[(t, i)].index[:10]),
}

sel_rows, chosen = [], {}
for task in TASKS:
    for sname, fn in STRATS.items():
        for i, (tr, va) in enumerate(FOLDS):
            cols = fn(tr, task, i)
            chosen.setdefault((task, sname), []).append(cols)
            pred, _ = fit_predict("lightgbm", task, tr, va, cols)
            sel_rows.append(dict(task=task, strategy=sname, fold=i, n_feat=len(cols), **score(task, va, pred)))

sel = (
    pl.DataFrame(sel_rows)
      .group_by(["task", "strategy"])
      .agg(n_feat=pl.col("n_feat").mean(), ratio=pl.col("ratio").mean(), ratio_std=pl.col("ratio").std())
      .sort(["task", "ratio"])
)
print(sel)

# how stable is the top-20 selection across folds?
for task in TASKS:
    counts = pd.Series([f for cols in chosen[(task, "top_20")] for f in cols]).value_counts()
    print(f"\n[{task}] features picked in all 5 folds:", list(counts[counts == 5].index))

shape: (8, 5)
┌──────┬─────────────┬────────┬──────────┬───────────┐
│ task ┆ strategy    ┆ n_feat ┆ ratio    ┆ ratio_std │
│ ---  ┆ ---         ┆ ---    ┆ ---      ┆ ---       │
│ str  ┆ str         ┆ f64    ┆ f64      ┆ f64       │
╞══════╪═════════════╪════════╪══════════╪═══════════╡
│ ret  ┆ all         ┆ 64.0   ┆ 1.003427 ┆ 0.004053  │
│ ret  ┆ top_20      ┆ 20.0   ┆ 1.00383  ┆ 0.003556  │
│ ret  ┆ corr_filter ┆ 54.4   ┆ 1.004437 ┆ 0.004339  │
│ ret  ┆ top_10      ┆ 10.0   ┆ 1.004602 ┆ 0.002763  │
│ vol  ┆ all         ┆ 64.0   ┆ 0.946069 ┆ 0.027177  │
│ vol  ┆ corr_filter ┆ 54.4   ┆ 0.946755 ┆ 0.027586  │
│ vol  ┆ top_20      ┆ 20.0   ┆ 0.948119 ┆ 0.025933  │
│ vol  ┆ top_10      ┆ 10.0   ┆ 0.949645 ┆ 0.026124  │
└──────┴─────────────┴────────┴──────────┴───────────┘

[ret] features picked in all 5 folds: ['mkt_lag1', 'lag_21', 'mkt_abs_lag1', 'lag_30', 'lag_8', 'lag_25', 'lag_9', 'lag_17']

[vol] features picked in all 5 folds: ['currency', 'roll_abs_30', 'ewm_vol_30', 'mkt_abs_

### Permutation importance (a second opinion)
Gain importance can favour noisy features. Permutation importance shuffles one feature at a time on held-out data and measures how much MAE gets worse. Shown for the last fold.

In [ ]:
for task in TASKS:
    tr, va = FOLDS[-1]
    _, m = fit_predict("lightgbm", task, tr, va, ALL)
    Xva = make_X_lgb(va, ALL)
    res = permutation_importance(m, Xva, va[f"y_{task}"].to_numpy(), n_repeats=5, random_state=42,
                                 scoring="neg_mean_absolute_error")
    pi = pd.Series(res.importances_mean, index=ALL).sort_values(ascending=False)
    print(f"[{task}] top permutation importances (MAE increase when shuffled):")
    print(pi.head(10).round(6).to_string(), "\n")

[ret] top permutation importances (MAE increase when shuffled):
mkt_lag1       0.000016
currency       0.000012
dow            0.000005
lag_11         0.000003
dom            0.000002
lag_17         0.000002
lag_1          0.000002
lag_2          0.000002
roll_mean_5    0.000002
lag_20         0.000001 

[vol] top permutation importances (MAE increase when shuffled):
roll_abs_30     0.000038
ewm_vol_30      0.000017
mkt_abs_lag1    0.000006
currency        0.000004
mkt_vol_5       0.000003
ewm_vol_10      0.000002
roll_mean_20    0.000002
dom             0.000002
mkt_lag1        0.000002
roll_mean_30    0.000002 



## 6. Hyperparameter tuning (Optuna, time-series CV objective)
LightGBM, searched on the training folds only, per track, using the feature set selected by the best strategy above. Caveat: the tuned CV score is slightly optimistic because the search saw the same folds. The honest check is the single test evaluation in section 8.

In [ ]:
def final_cols(task: str, strat: str) -> list:
    if strat == "all":
        return ALL
    if strat == "corr_filter":
        return corr_filter(train_df)
    k = int(strat.split("_")[1])
    return list(gain_ranking(train_df, task).index[:k])

best_strat = {t: sel.filter(pl.col("task") == t)["strategy"][0] for t in TASKS}
SELECTED = {t: final_cols(t, best_strat[t]) for t in TASKS}
print("best strategy:", best_strat, "| n features:", {t: len(c) for t, c in SELECTED.items()})

def cv_ratio(task, cols, params=None):
    return float(np.mean([score(task, va, fit_predict("lightgbm", task, tr, va, cols, params)[0])["ratio"]
                          for tr, va in FOLDS]))

def tune(task, cols, n_trials=20):
    def objective(trial):
        p = dict(
            num_leaves=trial.suggest_int("num_leaves", 7, 63),
            max_depth=trial.suggest_int("max_depth", 3, 10),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            n_estimators=trial.suggest_int("n_estimators", 100, 500),
            min_child_samples=trial.suggest_int("min_child_samples", 20, 300),
            subsample=trial.suggest_float("subsample", 0.5, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.3, 1.0),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        )
        return cv_ratio(task, cols, p)
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials)
    return study

TUNED = {}
for task in TASKS:
    t0 = time.time()
    default_ratio = cv_ratio(task, SELECTED[task])
    study = tune(task, SELECTED[task])
    TUNED[task] = study.best_params
    print(f"[{task}] CV ratio default={default_ratio:.4f} -> tuned={study.best_value:.4f} "
          f"({len(study.trials)} trials, {time.time() - t0:.0f}s)")
    print("   best params:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in study.best_params.items()})

best strategy: {'ret': 'all', 'vol': 'all'} | n features: {'ret': 64, 'vol': 64}
[ret] CV ratio default=1.0034 -> tuned=0.9999 (20 trials, 107s)
   best params: {'num_leaves': 15, 'max_depth': 3, 'learning_rate': 0.0165, 'n_estimators': 121, 'min_child_samples': 202, 'subsample': 0.6763, 'colsample_bytree': 0.9525, 'reg_lambda': 0.0162}
[vol] CV ratio default=0.9461 -> tuned=0.9433 (20 trials, 201s)
   best params: {'num_leaves': 9, 'max_depth': 8, 'learning_rate': 0.014, 'n_estimators': 485, 'min_child_samples': 77, 'subsample': 0.81, 'colsample_bytree': 0.422, 'reg_lambda': 0.0155}


## 7a. Direction classifier (up vs down)
Daily returns are split almost exactly 50/50 between up and down (flat days are about 0.5% and are dropped), so this is a **balanced** problem: no class weights, no resampling. The right yardsticks are accuracy and ROC-AUC against 50%. This is the same question the regression models answer implicitly through `dir_acc`, asked directly.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def dir_data(tr, va):
    tr2 = tr.filter(pl.col("y_ret") != 0).with_columns(label=(pl.col("y_ret") > 0).cast(pl.Int8))
    va2 = va.filter(pl.col("e_ret") != 0).with_columns(label=(pl.col("e_ret") > 0).cast(pl.Int8))
    return tr2, va2          # train labels from repaired returns, validation labels from raw returns

def make_dir_model(name):
    params = {k: v for k, v in LGB_BASE.items() if k != "objective"}
    if name == "lightgbm":
        return lgb.LGBMClassifier(objective="binary", **params), make_X_lgb
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.01, max_iter=500)), make_X_sk

dir_rows = []
for i, (tr, va) in enumerate(FOLDS):
    tr2, va2 = dir_data(tr, va)
    ytr, yva = tr2["label"].to_numpy(), va2["label"].to_numpy()
    maj_acc = yva.mean() if ytr.mean() >= 0.5 else 1 - yva.mean()      # always predict the training majority class
    for name in ("lightgbm", "logistic"):
        model, mk = make_dir_model(name)
        model.fit(mk(tr2, ALL), ytr)
        p = model.predict_proba(mk(va2, ALL))[:, 1]
        dir_rows.append(dict(model=name, fold=i, up_share_train=float(ytr.mean()), up_share_val=float(yva.mean()),
                             accuracy=float(accuracy_score(yva, (p >= 0.5).astype(int))),
                             majority_acc=float(maj_acc), roc_auc=float(roc_auc_score(yva, p))))

dirs = pl.DataFrame(dir_rows)
print(dirs.group_by("model").agg(pl.col(["up_share_train", "up_share_val", "accuracy", "majority_acc", "roc_auc"]).mean()))

shape: (2, 6)
┌──────────┬────────────────┬──────────────┬──────────┬──────────────┬──────────┐
│ model    ┆ up_share_train ┆ up_share_val ┆ accuracy ┆ majority_acc ┆ roc_auc  │
│ ---      ┆ ---            ┆ ---          ┆ ---      ┆ ---          ┆ ---      │
│ str      ┆ f64            ┆ f64          ┆ f64      ┆ f64          ┆ f64      │
╞══════════╪════════════════╪══════════════╪══════════╪══════════════╪══════════╡
│ lightgbm ┆ 0.498811       ┆ 0.495152     ┆ 0.509274 ┆ 0.499104     ┆ 0.515489 │
│ logistic ┆ 0.498811       ┆ 0.495152     ┆ 0.504351 ┆ 0.499104     ┆ 0.511435 │
└──────────┴────────────────┴──────────────┴──────────┴──────────────┴──────────┘


## 7b. Spike alert: the "large move" event (genuinely imbalanced)
Event = a day whose |return| is in the top 10% for that currency, with the threshold fitted from each fold's training part. About 90/10, so a plain classifier can score well by almost never predicting an event.

What we compare:
- `plain`: no imbalance handling
- `weighted`: `scale_pos_weight` = negatives / positives

Metrics: **PR-AUC** (compare it with the base rate; a random scorer's PR-AUC equals the base rate), ROC-AUC, **lift at top 10%** (precision when we flag the 10% highest scores, divided by the base rate), and F1 at the default 0.5 cut-off, to show what weighting does to thresholded behaviour. The baseline scorer is the 5-day rolling volatility.

SMOTE-style oversampling is deliberately not used: synthetic rows break the temporal structure of a time series.

In [ ]:
Q_EVENT = 0.90

def event_frame(tr, va):
    thr = tr.group_by("currency").agg(thr=pl.col("y_vol").quantile(Q_EVENT))
    tr = tr.join(thr, on="currency", how="left").with_columns(label=(pl.col("y_vol") > pl.col("thr")).cast(pl.Int8))
    va = (
        va.join(thr, on="currency", how="left")
          .filter(pl.col("thr").is_not_null())      # a currency with no training rows in this fold has no threshold
          .with_columns(label=(pl.col("e_vol") > pl.col("thr")).cast(pl.Int8))
    )
    return tr, va            # train labels from repaired returns, validation labels from raw returns

def cls_metrics(y, p):
    k = max(int(round(0.10 * len(y))), 1)
    top = np.argsort(-p)[:k]
    base = y.mean()
    return dict(
        base_rate=float(base),
        pr_auc=float(average_precision_score(y, p)),
        roc_auc=float(roc_auc_score(y, p)),
        lift_top10=float(y[top].mean() / base),
        f1_at_0_5=float(f1_score(y, (p >= 0.5).astype(int), zero_division=0)),
    )

def make_clf(weighted: bool, pos_rate: float):
    spw = (1 - pos_rate) / pos_rate if weighted else 1.0
    params = {k: v for k, v in LGB_BASE.items() if k != "objective"}
    return lgb.LGBMClassifier(objective="binary", scale_pos_weight=spw, **params)

ev_rows = []
for i, (tr, va) in enumerate(FOLDS):
    tr2, va2 = event_frame(tr, va)
    ytr, yva = tr2["label"].to_numpy(), va2["label"].to_numpy()
    for variant in ("plain", "weighted"):
        clf = make_clf(variant == "weighted", ytr.mean()).fit(make_X_lgb(tr2, ALL), ytr)
        p = clf.predict_proba(make_X_lgb(va2, ALL))[:, 1]
        ev_rows.append(dict(scorer=variant, fold=i, **cls_metrics(yva, p)))
    for b in ("roll_std_5", "ewm_vol_10"):
        ev_rows.append(dict(scorer=f"baseline:{b}", fold=i, **cls_metrics(yva, va2[b].to_numpy())))

ev = (
    pl.DataFrame(ev_rows)
      .group_by("scorer")
      .agg(pl.col(["base_rate", "pr_auc", "roc_auc", "lift_top10", "f1_at_0_5"]).mean())
      .sort("pr_auc", descending=True)
)
print(ev)

shape: (4, 6)
┌─────────────────────┬───────────┬──────────┬──────────┬────────────┬───────────┐
│ scorer              ┆ base_rate ┆ pr_auc   ┆ roc_auc  ┆ lift_top10 ┆ f1_at_0_5 │
│ ---                 ┆ ---       ┆ ---      ┆ ---      ┆ ---        ┆ ---       │
│ str                 ┆ f64       ┆ f64      ┆ f64      ┆ f64        ┆ f64       │
╞═════════════════════╪═══════════╪══════════╪══════════╪════════════╪═══════════╡
│ weighted            ┆ 0.120383  ┆ 0.238261 ┆ 0.68697  ┆ 2.808244   ┆ 0.286356  │
│ plain               ┆ 0.120383  ┆ 0.233627 ┆ 0.68544  ┆ 2.730349   ┆ 0.036292  │
│ baseline:ewm_vol_10 ┆ 0.120383  ┆ 0.226183 ┆ 0.662508 ┆ 2.52499    ┆ 0.0       │
│ baseline:roll_std_5 ┆ 0.120383  ┆ 0.205754 ┆ 0.640308 ┆ 2.351986   ┆ 0.0       │
└─────────────────────┴───────────┴──────────┴──────────┴────────────┴───────────┘


## 8. Final evaluation on the test window (one look)
Per regression track, two models are scored on the test window: (a) the default LightGBM on all candidate features, and (b) the final pipeline (selected features + tuned parameters). Then the direction classifier and the event classifier. We do not iterate on these numbers.

In [ ]:
final_rows = []
for task in TASKS:
    for label, cols, params in [("default_all_features", ALL, None),
                                ("selected_and_tuned", SELECTED[task], TUNED[task])]:
        pred, _ = fit_predict("lightgbm", task, train_df, test_df, cols, params)
        final_rows.append(dict(task=task, model=label, n_feat=len(cols), **score(task, test_df, pred)))
final = pl.DataFrame(final_rows)
print(final.select(["task", "model", "n_feat", "mae", "mae_base", "ratio", "dir_acc"]))

tr2, te2 = event_frame(train_df, test_df)
ytr, yte = tr2["label"].to_numpy(), te2["label"].to_numpy()
ev_test = []
for variant in ("plain", "weighted"):
    clf = make_clf(variant == "weighted", ytr.mean()).fit(make_X_lgb(tr2, ALL), ytr)
    ev_test.append(dict(scorer=variant, **cls_metrics(yte, clf.predict_proba(make_X_lgb(te2, ALL))[:, 1])))
ev_test.append(dict(scorer="baseline:roll_std_5", **cls_metrics(yte, te2["roll_std_5"].to_numpy())))
print(pl.DataFrame(ev_test))

tr3, te3 = dir_data(train_df, test_df)
ytr3, yte3 = tr3["label"].to_numpy(), te3["label"].to_numpy()
dmodel, mk = make_dir_model("lightgbm")
dmodel.fit(mk(tr3, ALL), ytr3)
pd3 = dmodel.predict_proba(mk(te3, ALL))[:, 1]
maj = yte3.mean() if ytr3.mean() >= 0.5 else 1 - yte3.mean()
print("DIRECTION test | accuracy:", round(float(accuracy_score(yte3, (pd3 >= 0.5).astype(int))), 4),
      "| majority-class accuracy:", round(float(maj), 4), "| ROC-AUC:", round(float(roc_auc_score(yte3, pd3)), 4))

shape: (4, 7)
┌──────┬──────────────────────┬────────┬──────────┬──────────┬──────────┬──────────┐
│ task ┆ model                ┆ n_feat ┆ mae      ┆ mae_base ┆ ratio    ┆ dir_acc  │
│ ---  ┆ ---                  ┆ ---    ┆ ---      ┆ ---      ┆ ---      ┆ ---      │
│ str  ┆ str                  ┆ i64    ┆ f64      ┆ f64      ┆ f64      ┆ f64      │
╞══════╪══════════════════════╪════════╪══════════╪══════════╪══════════╪══════════╡
│ ret  ┆ default_all_features ┆ 64     ┆ 0.003303 ┆ 0.003299 ┆ 1.00118  ┆ 0.5064   │
│ ret  ┆ selected_and_tuned   ┆ 64     ┆ 0.003296 ┆ 0.003299 ┆ 0.999036 ┆ 0.515472 │
│ vol  ┆ default_all_features ┆ 64     ┆ 0.002096 ┆ 0.002278 ┆ 0.920128 ┆ null     │
│ vol  ┆ selected_and_tuned   ┆ 64     ┆ 0.002095 ┆ 0.002278 ┆ 0.919722 ┆ null     │
└──────┴──────────────────────┴────────┴──────────┴──────────┴──────────┴──────────┘
shape: (3, 6)
┌─────────────────────┬───────────┬──────────┬──────────┬────────────┬───────────┐
│ scorer              ┆ base_rate ┆ pr_

## 9. Conclusions (fill in after running)
- Model comparison: best model per track ____ ; does any beat the baseline on `ret`? ____ ; on `vol`? ____
- Feature selection: best strategy per track ____ ; features that survive in every fold ____
- Tuning: CV ratio default -> tuned ____ ; did it change the picture? ____
- Direction (balanced, no weighting): accuracy ____ vs majority-class ____ ; ROC-AUC ____
- Spike alert (imbalanced): PR-AUC vs base rate ____ ; effect of weighting on F1 ____ ; lift at top 10% ____
- Final test (one look): `ret` ratio ____ ; `vol` ratio ____ ; direction accuracy ____ ; event PR-AUC ____
- Decision for Phase 3: target = ____ ; model = ____ ; features = ____ ; params saved to MLflow
- Caveats: 7 prototype currencies only; one split; `vol` track chosen after a test peek